# Module 02 — Write Problems (Colab)

A real agent (local Ollama LLM) files a regulatory obligation into a real Postgres database. Run it twice and it files the same obligation twice — because the model words it differently each time. You'll fix that.

**Run the cells top to bottom.** The setup cell takes a few minutes the first time (it installs Postgres + Ollama and pulls the model).

## 1. Get the code and set up the environment

In [ ]:
!git clone https://github.com/sanbhaumik/workshop-designing-data-infra-for-ai-agents.git repo
%cd repo

In [ ]:
# Installs Postgres + Ollama + the model + Python deps. Slow the first time.
!bash setup.sh

In [ ]:
import os
os.environ['NOVA_LLM'] = 'ollama'
os.environ['OLLAMA_MODEL'] = 'llama3.2:1b'
os.environ['DATABASE_URL'] = 'postgresql://postgres@localhost:5432/nova'
print('environment configured')

In [ ]:
!python preflight.py

## 2. Watch the agent fail

Two runs (a retry). Watch the two REASON lines — same obligation, different wording — and the regulator receiving **two** filings.

In [ ]:
!python modules/02_write_path/naive.py

See the duplicate rows in the **real database** with SQL:

In [ ]:
!psql "$DATABASE_URL" -c "SELECT client_id, left(obligation_text,60) AS obligation FROM filings;"

## 3. Fix it

The cell below is `your_fix.py`. Right now `obligation_identity` keys on `obligation_text`, which changes every run. **Edit the last line** so the key is derived from the agent's stable intent — `client_id` and `source_doc` — then re-run this cell to save it.

Hint: `return hashlib.sha256(f"{client_id}|{source_doc}".encode("utf-8")).hexdigest()`

In [ ]:
%%writefile modules/02_write_path/your_fix.py
import hashlib


def obligation_identity(client_id: str, source_doc: str, obligation_text: str) -> str:
    """Return a STABLE idempotency key identifying this obligation."""
    # TODO: key on the agent's INTENT (client_id, source_doc), NOT obligation_text.
    return hashlib.sha256(obligation_text.encode("utf-8")).hexdigest()

In [ ]:
!python -m pytest modules/02_write_path/test_write.py -v

## 4. See it land

Before/after: the naive identity files twice; your identity files once.

In [ ]:
!python modules/02_write_path/compare.py